# 07. 🏁 Day 4 통합 미션 · 카페 주문 데이터 분석

오늘 배운 **`.str`(정제) → `.dt`(날짜 파생) → `groupby`(요약)** 를 **스스로 이어서** 풀어봅니다.

새 지점의 주문 데이터 `주문로그_미션.csv` 가 주어집니다. 앞 노트북과 **구조는 같지만 내용은 다릅니다.**
힌트 주석을 참고해 직접 완성하세요. 정답이 막히면 앞의 `01·02·03` 노트북을 참고해도 좋아요.

> 🎯 최종 목표 : 지저분한 원본 → 깨끗한 분석용 표 → **"이 지점, 뭐가 잘 팔리고 언제 붐비나?"** 에 답하기

## 0단계 · 데이터 불러오기 (제공)

In [ ]:
# [제공] 데이터 로드
import pandas as pd

df = pd.read_csv('data/주문로그_미션.csv', encoding='utf-8-sig')
print('원본 크기:', df.shape)
df.head()

## 🎯 미션 1 · 문자열 정제 (`.str`)

- `상품명` : 앞뒤 공백 제거 + `ICE`/`ice` 를 `아이스` 로 통일
- `고객이메일` : 앞뒤 공백 제거 + 소문자로 통일

In [ ]:
# [미션 1] 문자열 정제
#  - df['상품명']   : .str.strip() 후 .str.replace 로 ICE/ice → 아이스
#  - df['고객이메일'] : .str.strip().str.lower()
df['상품명']   = df['상품명'].str.strip().str.replace('ICE', '아이스').str.replace('ice', '아이스')
df['고객이메일'] = df['고객이메일'].str.strip().str.lower()
df[['상품명', '고객이메일']].head()

## 🎯 미션 2 · 문자열에서 새 컬럼 만들기 (`.str`)

- `도메인` : 이메일에서 `@` 뒤쪽
- `시도` : 주소의 첫 단어
- `가격_숫자` : `'3,800원'` → `3800` (정수)
- `매출` : 가격_숫자 × 수량

In [ ]:
# [미션 2] 파생 컬럼 (문자열)
df['도메인']   = df['고객이메일'].str.split('@').str[1]
df['시도']     = df['주소'].str.split(' ').str[0]
df['가격_숫자'] = df['가격'].str.replace(',', '').str.replace('원', '').astype(int)
df['매출']     = df['가격_숫자'] * df['수량']
df[['도메인', '시도', '가격_숫자', '매출']].head()

## 🎯 미션 3 · 날짜 다루기 (`.dt`)

- `주문일시` 를 날짜형으로 변환(`pd.to_datetime`)
- `시` 컬럼(`.dt.hour`) 만들기

In [ ]:
# [미션 3] 날짜 변환 + 시(hour) 파생
df['주문일시'] = pd.to_datetime(df['주문일시'])
df['시'] = df['주문일시'].dt.hour
df[['주문일시', '시']].head()

## 🧩 요일·시간대 파생 (제공)
요일(한글)·시간대는 어제처럼 제공합니다. 실행만 하세요.

In [ ]:
# [제공] 요일 + 시간대
df['주문일시'] = pd.to_datetime(df['주문일시'])   # 안전하게 한 번 더
요일한글 = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
df['요일'] = df['주문일시'].dt.dayofweek.map(요일한글)
df['요일'] = pd.Categorical(df['요일'], categories=['월','화','수','목','금','토','일'], ordered=True)
df['시간대'] = pd.cut(df['주문일시'].dt.hour, bins=[0, 12, 17, 24],
                    labels=['오전', '오후', '저녁'], right=False)
df[['주문일시', '요일', '시간대']].head()

## 🎯 미션 4 · `groupby` 로 3가지 질문에 답하기

아래 세 가지를 각각 구해보세요.
1. **상품별 총매출** (많은 순)
2. **요일별 주문 건수**
3. **시간대별 평균 매출**

In [ ]:
# [미션 4-1] 상품별 총매출 (내림차순)
df.groupby('상품명')['매출'].sum().sort_values(ascending=False)

In [ ]:
# [미션 4-2] 요일별 주문 건수
df.groupby('요일').size()

In [ ]:
# [미션 4-3] 시간대별 평균 매출
df.groupby('시간대')['매출'].mean().round(0)

## ✍️ 미션 5 · 인사이트 3줄 (서술형)

위 결과를 보고, 이 지점 사장님께 보고하듯 **한글로 3줄** 적어보세요. 예)
- 매출 1위 메뉴는 ○○이고, 전체 매출의 상당 부분을 차지한다.
- 주문은 ○요일 · ○시간대에 가장 많다 → 그 시간에 인력을 더 배치하면 좋겠다.
- ○○ 메뉴는 객단가가 높으니 세트/추천으로 밀어보면 좋겠다.

> (아래 칸을 더블클릭해 직접 작성하세요)

1.
2.
3.

---
### ✅ 미션 완료 체크
- [ ] 문자열 정제(strip·replace·lower)를 했다
- [ ] `.str.split` 으로 도메인·시도를 뽑았다
- [ ] 가격 텍스트를 숫자로 바꿔 매출을 계산했다
- [ ] `to_datetime` 후 `.dt` 로 날짜 컬럼을 만들었다
- [ ] `groupby` 로 3가지 질문에 답했다
- [ ] 인사이트 3줄을 적었다

수고했습니다! 🎉 이걸 해냈다면 **"지저분한 실무 데이터 → 요약·인사이트"** 한 사이클을 스스로 돈 겁니다.